# Quantum clustering with train and test splits

Clustering is usually explored on the whole dataset, but a model that can **assign unseen samples** lets us check whether the structure it found generalises. In this example we fit Probabilistic Quantum Clustering on a train split and assign the test samples together with their membership probabilities.

It needs the optional clustering dependencies: `pip install "falcondale[clustering]"`.

## A synthetic cohort

We simulate 240 samples with 30 ordinal measurements (0 to 5 scales). Four profiles are planted: three of them raise a different block of measurements, the last one stays mild everywhere. Knowing the true profile lets us score the clustering, but it is never shown to the model.

In [1]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(101)
samples, features, profiles = 240, 30, 4
profile = rng.integers(0, profiles, samples)

means = np.full((profiles, features), 0.8)
block = features // (profiles - 1)
for p in range(profiles - 1):
    means[p, p * block : (p + 1) * block] = 3.8

values = np.clip(np.rint(rng.normal(means[profile], 1.6)), 0, 5).astype(int)
data = pd.DataFrame(values, columns=[f"f{i:02d}" for i in range(features)])
data.head()

,f00,f01,f02,f03,f04,f05,f06,f07,f08,f09,...,f20,f21,f22,f23,f24,f25,f26,f27,f28,f29
0,0,1,2,2,0,0,2,2,0,1,...,2,3,0,0,2,0,1,1,0,3
1,1,0,1,1,0,0,0,0,2,3,...,0,2,0,0,0,4,0,2,4,0
2,0,0,0,3,0,2,2,1,2,0,...,1,5,5,3,4,5,3,1,4,2
3,0,1,2,4,1,1,2,1,0,0,...,0,1,0,1,0,2,0,3,2,0
4,4,4,5,3,5,3,1,4,4,5,...,0,1,0,1,0,3,0,0,1,0


In [2]:
from sklearn.model_selection import train_test_split

train, test, train_profile, test_profile = train_test_split(
    data, profile, test_size=0.3, random_state=101, stratify=profile
)
len(train), len(test)

(168, 72)

## Fitting on the train split

`make_clustering_pipeline` chains median imputation, standard scaling, PCA and `QuantumClustering`. All of them are fitted on the train samples only, so nothing from the test split leaks into the model.

The quantum clustering step searches a grid of bandwidths. For each one, samples descend the quantum potential to its wells, and those wells can be merged by energy into coarser partitions. Among the admissible candidates (2 to 8 clusters, no tiny cluster) the one with the best training silhouette is kept.

In [3]:
from falcondale.clustering import make_clustering_pipeline

pipeline = make_clustering_pipeline(components=6, random_state=101)
pipeline.fit(train)

model = pipeline[-1]
print(f"Selected bandwidth {model.bandwidth_}, stage {model.stage_}, {model.n_clusters_} clusters")
print(f"Training silhouette: {model.silhouette_:.3f}")

Selected bandwidth ('knn', 0.2), stage base, 4 clusters
Training silhouette: 0.463


Every candidate of the search is recorded, which helps understanding why a partition was chosen. Very small bandwidths split the data into many wells, the largest ones merge everything.

In [4]:
pd.DataFrame(model.candidates_)[["bandwidth", "stage", "k", "silhouette", "admissible", "steps"]]

,bandwidth,stage,k,silhouette,admissible,steps
0,knn:0.05,base,124,0.039238,False,156
1,knn:0.1,base,31,0.025408,False,138
2,knn:0.2,base,4,0.462648,True,247
3,knn:0.2,energy_1,3,0.424912,True,247
4,knn:0.2,energy_2,2,0.308258,True,247
5,knn:0.2,energy_3,1,NaN,False,247
6,knn:0.3,base,4,0.451543,True,162
7,knn:0.3,energy_1,3,0.403111,True,162
8,knn:0.3,energy_2,2,0.285955,True,162
9,knn:0.3,energy_3,1,NaN,False,162


## Assigning the test samples

New samples are transformed with the preprocessing fitted on the train split and assigned to the clusters found. The adjusted Rand index compares the assignment with the planted profiles (1 is a perfect match, 0 is what random labels would get).

In [5]:
from sklearn.metrics import adjusted_rand_score

test_labels = pipeline.predict(test)
probabilities = pipeline.predict_proba(test)

print(f"Train ARI: {adjusted_rand_score(train_profile, model.labels_):.3f}")
print(f"Test ARI: {adjusted_rand_score(test_profile, test_labels):.3f}")
pd.crosstab(test_profile, test_labels, rownames=["planted profile"], colnames=["cluster"])

Train ARI: 1.000
Test ARI: 1.000


cluster,0,1,2,3
planted profile,,,,
0,0,19,0,0
1,20,0,0,0
2,0,0,0,16
3,0,0,17,0


The probabilities tell confident assignments from samples lying between clusters. They are the membership weights of APQC rather than calibrated probabilities, but they rank samples by how clearly they belong to their cluster.

In [6]:
confidence = pd.Series(probabilities.max(axis=1), name="confidence")
confidence.describe()

count    72.000000
mean      0.962631
std       0.053044
min       0.731612
25%       0.955770
50%       0.983273
75%       0.990488
max       0.998120
Name: confidence, dtype: float64

## Going further

* Narrow or widen the search with `bandwidths=["knn:0.2", "scalar:0.4"]`.
* Change the descent budget with `steps`, `patience` and `learning_rate`.
* Adapt the admissibility rules with `min_clusters`, `max_clusters`, `min_cluster_size` and `min_cluster_fraction`.
* Without splits, `Project(data).cluster("pqc")` clusters a whole dataset, no target needed.